# Investigating Factors Contributing to High Star Ratings on Yelp

- **Author**: Ervin Pangilinan
- COSC 526: Data Mining & Analytics Spring 2025


## Import Libraries

In [412]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from pyspark import SparkContext
from pyspark.sql.functions import col, when, trim

## Data Preprocessing

In [413]:
# Making Spark Context
sc = SparkContext.getOrCreate()

Once we load the raw data, we will need to extract the relevant features for our analysis.

In [414]:
business_df = pd.read_json('yelp_academic_dataset_business.json', lines=True)
# checkin_df = pd.read_json('yelp_academic_dataset_checkin.json', lines=True)
# tip_df = pd.read_json('yelp_academic_dataset_tip.json', lines=True)

# Drop is_open column
business_df = business_df.drop(columns=['is_open', 'business_id'])

business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,attributes,categories,hours
0,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."
5,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
6,Famous Footwear,"8522 Eager Road, Dierbergs Brentwood Point",Brentwood,MO,63144,38.627695,-90.340465,2.5,13,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Sporting Goods, Fashion, Shoe Stores, Shopping...","{'Monday': '0:0-0:0', 'Tuesday': '10:0-18:0', ..."
7,Temple Beth-El,400 Pasadena Ave S,St. Petersburg,FL,33707,27.766590,-82.732983,3.5,5,None,"Synagogues, Religious Organizations","{'Monday': '9:0-17:0', 'Tuesday': '9:0-17:0', ..."
8,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
9,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."


In [415]:
business_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   name          150346 non-null  object 
 1   address       150346 non-null  object 
 2   city          150346 non-null  object 
 3   state         150346 non-null  object 
 4   postal_code   150346 non-null  object 
 5   latitude      150346 non-null  float64
 6   longitude     150346 non-null  float64
 7   stars         150346 non-null  float64
 8   review_count  150346 non-null  int64  
 9   attributes    136602 non-null  object 
 10  categories    150243 non-null  object 
 11  hours         127123 non-null  object 
dtypes: float64(3), int64(1), object(8)
memory usage: 13.8+ MB


After loading the data, we see that we will need to filter the data to only include restaurants that are in the data. Next, we'll have to do additional preprocessing to clean the data and only include restaurants.

In [416]:
# Filter for businesses with the restaurant attribute
business_df = business_df[business_df['categories'].str.contains('Restaurant', na=False)]
business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,attributes,categories,hours
3,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
5,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
8,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
9,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."
11,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks","{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."
12,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"{'RestaurantsReservations': 'False', 'Restaura...","American (Traditional), Restaurants, Diners, B...","{'Monday': '6:0-22:0', 'Tuesday': '6:0-22:0', ..."
14,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"{'OutdoorSeating': 'False', 'RestaurantsGoodFo...","Food, Delis, Italian, Bakeries, Restaurants","{'Monday': '10:0-18:0', 'Tuesday': '10:0-20:0'..."
15,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese","{'Tuesday': '13:30-22:0', 'Wednesday': '13:30-..."
19,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"{'NoiseLevel': 'u'quiet'', 'GoodForMeal': '{'d...","Korean, Restaurants","{'Monday': '11:30-20:30', 'Tuesday': '11:30-20..."
20,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"{'OutdoorSeating': 'False', 'Caters': 'True', ...","Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...","{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."


#### Dealing with Missing Values

Next, let's check for missing values in the data. We will need to drop any rows that have missing values in the features we are interested in. We will also need to convert the data types of some of the columns to make them easier to work with.

In [417]:
# Replace cells with only blank spaces in any column with NaN and check for missing data
business_df = business_df.applymap(lambda x: np.nan if isinstance(x, str) and not x.strip() else x)
missing_data = business_df[business_df.isnull().any(axis=1)]
missing_data.head(10)

/var/folders/qd/cq8k89k14gs26x42sg8fmrxc0000gn/T/ipykernel_3951/3715250282.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  business_df = business_df.applymap(lambda x: np.nan if isinstance(x, str) and not x.strip() else x)


,name,address,city,state,postal_code,latitude,longitude,stars,review_count,attributes,categories,hours
8,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
11,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks","{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."
24,Super Dog,1160 Gallatin Pike S,Nashville,TN,37115,36.248570,-86.719986,4.0,6,"{'RestaurantsReservations': 'False', 'Restaura...","Hot Dogs, Restaurants",None
36,Cheeseburger In Paradise,116 N Pottstown Pike,Exton,PA,19341,40.029962,-75.630607,2.5,20,"{'NoiseLevel': 'u'average'', 'HasTV': 'True', ...","Restaurants, Burgers",None
62,Village Tap Room,838 Broad Ripple Ave,Indianapolis,IN,46220,39.869911,-86.143577,2.5,23,"{'Alcohol': 'u'none'', 'BestNights': '{'monday...","Gastropubs, Cocktail Bars, Beer Bar, Bars, Res...",None
64,Doc Magrogan's Oyster House - West Chester,117 E Gay St,West Chester,PA,19380,39.961542,-75.603604,3.0,114,"{'WiFi': 'u'no'', 'RestaurantsAttire': 'u'casu...","Seafood, Restaurants, Bars, Nightlife, Cocktai...",None
73,Sharky's Sports Bar & Grill,820 N Black Horse Pike,Williamstown,NJ,08094,39.696801,-74.999821,2.5,29,"{'Alcohol': 'u'full_bar'', 'RestaurantsGoodFor...","American (Traditional), Bars, Nightlife, Sport...",None
78,Gavi Italian Restaurant,"7401 N La Cholla Blvd, Ste 146",Tucson,AZ,85707,32.221667,-110.925833,3.5,9,"{'RestaurantsPriceRange2': '2', 'RestaurantsRe...","Italian, Restaurants",None
83,Super Sushi Kyo Hin,2501 Mt Holly Rd 245,Burlington,NJ,08016,40.041629,-74.825821,3.5,6,"{'RestaurantsReservations': 'True', 'Restauran...","Restaurants, Japanese, Sushi Bars, Asian Fusion",None
93,Impasto,NaN,Tampa,FL,33611,27.890814,-82.502346,5.0,5,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Restaurants, Italian, Food Trucks, Food","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W..."


In [418]:
missing_data_count = business_df.isnull().sum()
print(f"Missing data count:\n{missing_data_count}")

# Total number of rows with missing data
total_missing_data = missing_data.shape[0]
print(f"\nTotal number of rows with missing data: {total_missing_data}")

Missing data count:
name               0
address          443
city               0
state              0
postal_code       21
latitude           0
longitude          0
stars              0
review_count       0
attributes       566
categories         0
hours           7279
dtype: int64

Total number of rows with missing data: 7800


We see that there out of the 52286 samples in our dataset, 7800 of them have missing values. Further investigation is needed to see which features have missing values and how we can handle them.

In [419]:
# Calculate the percentage of missing values in each column
missing_percentage = business_df.isnull().mean()
missing_percentage

name            0.000000
address         0.008473
city            0.000000
state           0.000000
postal_code     0.000402
latitude        0.000000
longitude       0.000000
stars           0.000000
review_count    0.000000
attributes      0.010825
categories      0.000000
hours           0.139215
dtype: float64

##### Missing Attributes
Since samples with missing attributes only make up about 1% of this reduced dataset, we can drop them.

In [420]:
# Drop rows with missing attributes
business_df = business_df.dropna(subset=['attributes'])

# Check the number of rows after dropping missing values
print(f"Number of rows after dropping missing values: {business_df.shape[0]}")

Number of rows after dropping missing values: 51720


##### Missing Hours and Transforming the Feature

In this dataset, the hours are listed as a dictionary with the days of the week as keys and the hours of operation as values. We will need to convert this to a more usable format. We will transform this feature into hours open during the week and hours open on weekends. 

In [421]:
# Transform the hours column into 2 columns: number of hours open during weekdays and weekends
def transform_hours(hours):
    if not isinstance(hours, dict):
        return pd.Series([np.nan, np.nan])
    weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    weekend = ['Saturday', 'Sunday']
    weekday_hours = sum([len(v) for k, v in hours.items() if k in weekdays])
    weekend_hours = sum([len(v) for k, v in hours.items() if k in weekend])
    return pd.Series([weekday_hours, weekend_hours])

if 'hours' in business_df.columns:
    business_df[['weekday_hours', 'weekend_hours']] = business_df['hours'].apply(transform_hours)
    business_df = business_df.drop(columns=['hours'])

# Check the first few rows of the transformed DataFrame
business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,attributes,categories,weekday_hours,weekend_hours
3,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...",40.0,16.0
5,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...",38.0,16.0
8,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",NaN,NaN
9,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",39.0,16.0
11,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks",45.0,17.0
12,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"{'RestaurantsReservations': 'False', 'Restaura...","American (Traditional), Restaurants, Diners, B...",40.0,16.0
14,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"{'OutdoorSeating': 'False', 'RestaurantsGoodFo...","Food, Delis, Italian, Bakeries, Restaurants",45.0,9.0
15,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese",40.0,20.0
19,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"{'NoiseLevel': 'u'quiet'', 'GoodForMeal': '{'d...","Korean, Restaurants",55.0,11.0
20,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"{'OutdoorSeating': 'False', 'Caters': 'True', ...","Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",40.0,17.0


We don't want to drop the rows with missing values in the hours column because it made up 14% of our dataset. Instead, we will use the median value to fill in the missing values since samples that had missing hours values will in turn have missing values in the hours open during the week and hours open on weekends.

In [422]:
# Fill in missing values in the weekday_hours and weekend_hours columns with with the median value of the column
weekday_hours_mode = business_df['weekday_hours'].median()
weekend_hours_mode = business_df['weekend_hours'].median()
business_df['weekday_hours'] = business_df['weekday_hours'].fillna(weekday_hours_mode)
business_df['weekend_hours'] = business_df['weekend_hours'].fillna(weekend_hours_mode)

# Check the first few rows of the DataFrame after filling missing values
business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,attributes,categories,weekday_hours,weekend_hours
3,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...",40.0,16.0
5,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...",38.0,16.0
8,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",43.0,18.0
9,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",39.0,16.0
11,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks",45.0,17.0
12,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"{'RestaurantsReservations': 'False', 'Restaura...","American (Traditional), Restaurants, Diners, B...",40.0,16.0
14,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"{'OutdoorSeating': 'False', 'RestaurantsGoodFo...","Food, Delis, Italian, Bakeries, Restaurants",45.0,9.0
15,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese",40.0,20.0
19,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"{'NoiseLevel': 'u'quiet'', 'GoodForMeal': '{'d...","Korean, Restaurants",55.0,11.0
20,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"{'OutdoorSeating': 'False', 'Caters': 'True', ...","Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",40.0,17.0


#### Flattening Raw Data for Features

In [423]:
# Flatten the DataFrame completely
def flatten_column(df, col):
    # Expand the first-level attributes dictionary
    df = df.copy()
    attributes_expanded = pd.json_normalize(df[col])

    # Merge carefully by resetting indices
    df = df.reset_index(drop=True)
    attributes_expanded = attributes_expanded.reset_index(drop=True)
    df = pd.concat([df, attributes_expanded], axis=1)
    
    # Drop the original attributes column
    df = df.drop(columns=[col])

    return df

business_df = flatten_column(business_df, 'attributes')
business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,categories,...,BestNights,BYOB,Corkage,BYOBCorkage,RestaurantsCounterService,Open24Hours,AgesAllowed,DietaryRestrictions,HairSpecializesIn,AcceptsInsurance
0,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"Pubs, Restaurants, Italian, Bars, American (Tr...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"Vietnamese, Food, Restaurants, Food Trucks",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"American (Traditional), Restaurants, Diners, B...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"Food, Delis, Italian, Bakeries, Restaurants",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"Sushi Bars, Restaurants, Japanese",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"Korean, Restaurants",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Let's check for missing data again.

In [424]:
# Check percentage of missing values in the flattened DataFrame
missing_percentage = business_df.isnull().mean()
print(f"Missing percentage in flattened DataFrame:\n{missing_percentage}")

Missing percentage in flattened DataFrame:
name                          0.000000
address                       0.007966
city                          0.000000
state                         0.000000
postal_code                   0.000387
latitude                      0.000000
longitude                     0.000000
stars                         0.000000
review_count                  0.000000
categories                    0.000000
weekday_hours                 0.000000
weekend_hours                 0.000000
RestaurantsDelivery           0.078538
OutdoorSeating                0.156226
BusinessAcceptsCreditCards    0.118852
BusinessParking               0.117807
BikeParking                   0.313708
RestaurantsPriceRange2        0.139346
RestaurantsTakeOut            0.058063
ByAppointmentOnly             0.935247
WiFi                          0.270128
Alcohol                       0.226991
Caters                        0.331535
RestaurantsAttire             0.246094
RestaurantsReservatio

We'll drop the columns where the percentage of missing values is greater than 50%.

In [425]:
# Drop columns with more than 50% missing values
missing_percentage = business_df.isnull().mean()
columns_to_drop = missing_percentage[missing_percentage >= 0.5].index
business_df = business_df.drop(columns=columns_to_drop)

# Check the number of rows and columns after dropping columns
print(f"Number of rows and columns after dropping columns: {business_df.shape}")

missing_percentage = business_df.isnull().mean()
print(f"Missing percentage in flattened DataFrame:\n{missing_percentage}")

business_df.head(10)

Number of rows and columns after dropping columns: (51720, 30)
Missing percentage in flattened DataFrame:
name                          0.000000
address                       0.007966
city                          0.000000
state                         0.000000
postal_code                   0.000387
latitude                      0.000000
longitude                     0.000000
stars                         0.000000
review_count                  0.000000
categories                    0.000000
weekday_hours                 0.000000
weekend_hours                 0.000000
RestaurantsDelivery           0.078538
OutdoorSeating                0.156226
BusinessAcceptsCreditCards    0.118852
BusinessParking               0.117807
BikeParking                   0.313708
RestaurantsPriceRange2        0.139346
RestaurantsTakeOut            0.058063
WiFi                          0.270128
Alcohol                       0.226991
Caters                        0.331535
RestaurantsAttire             0.2460

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,categories,...,Alcohol,Caters,RestaurantsAttire,RestaurantsReservations,Ambience,GoodForKids,RestaurantsGoodForGroups,HasTV,NoiseLevel,GoodForMeal
0,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",...,u'none',True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",...,u'none',False,u'casual',False,None,True,True,True,NaN,NaN
2,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"Pubs, Restaurants, Italian, Bars, American (Tr...",...,u'full_bar',True,u'casual',False,"{'romantic': False, 'intimate': False, 'touris...",True,True,True,u'average',NaN
3,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",...,u'none',False,'casual',False,NaN,True,False,True,NaN,NaN
4,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"Vietnamese, Food, Restaurants, Food Trucks",...,'none',NaN,NaN,False,"{'touristy': False, 'hipster': False, 'romanti...",NaN,NaN,NaN,NaN,"{'dessert': False, 'latenight': False, 'lunch'..."
5,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"American (Traditional), Restaurants, Diners, B...",...,'none',NaN,'casual',False,"{'touristy': None, 'hipster': None, 'romantic'...",True,True,True,NaN,"{'dessert': False, 'latenight': False, 'lunch'..."
6,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"Food, Delis, Italian, Bakeries, Restaurants",...,u'none',True,u'casual',False,"{'romantic': False, 'intimate': False, 'touris...",True,False,True,u'average',"{'dessert': False, 'latenight': False, 'lunch'..."
7,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"Sushi Bars, Restaurants, Japanese",...,'full_bar',NaN,'casual',True,"{'touristy': False, 'hipster': False, 'romanti...",False,True,False,u'average',"{'dessert': True, 'latenight': None, 'lunch': ..."
8,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"Korean, Restaurants",...,u'none',True,u'casual',False,"{'touristy': False, 'hipster': False, 'romanti...",True,False,True,u'quiet',"{'dessert': False, 'latenight': False, 'lunch'..."
9,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",...,u'beer_and_wine',True,u'casual',False,"{'romantic': False, 'intimate': False, 'classy...",False,True,False,u'average',NaN


#### Cleaning Usable Features
We will also need to convert the data types of some of the columns to make them easier to work with.

In [426]:
bool_columns = [
    'RestaurantsDelivery', 'OutdoorSeating', 'BusinessAcceptsCreditCards',
    'RestaurantsReservations', 'Caters', 'BikeParking', 'RestaurantsTakeOut',
    'RestaurantsGoodForGroups', 'GoodForKids', 'HasTV'
]

for col in bool_columns:
    business_df[col] = business_df[col].replace({None: False})
    business_df[col] = business_df[col].astype(bool)

missing_percentage = business_df.isnull().mean()
print(f"Missing percentage in flattened DataFrame:\n{missing_percentage}")

business_df.head(10)

Missing percentage in flattened DataFrame:
name                          0.000000
address                       0.007966
city                          0.000000
state                         0.000000
postal_code                   0.000387
latitude                      0.000000
longitude                     0.000000
stars                         0.000000
review_count                  0.000000
categories                    0.000000
weekday_hours                 0.000000
weekend_hours                 0.000000
RestaurantsDelivery           0.000000
OutdoorSeating                0.000000
BusinessAcceptsCreditCards    0.000000
BusinessParking               0.117807
BikeParking                   0.000000
RestaurantsPriceRange2        0.139346
RestaurantsTakeOut            0.000000
WiFi                          0.270128
Alcohol                       0.226991
Caters                        0.000000
RestaurantsAttire             0.246094
RestaurantsReservations       0.000000
Ambience             

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,categories,...,Alcohol,Caters,RestaurantsAttire,RestaurantsReservations,Ambience,GoodForKids,RestaurantsGoodForGroups,HasTV,NoiseLevel,GoodForMeal
0,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",...,u'none',True,NaN,False,NaN,False,False,False,NaN,NaN
1,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",...,u'none',True,u'casual',True,None,True,True,True,NaN,NaN
2,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"Pubs, Restaurants, Italian, Bars, American (Tr...",...,u'full_bar',True,u'casual',True,"{'romantic': False, 'intimate': False, 'touris...",True,True,True,u'average',NaN
3,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",...,u'none',True,'casual',True,NaN,True,True,True,NaN,NaN
4,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"Vietnamese, Food, Restaurants, Food Trucks",...,'none',False,NaN,True,"{'touristy': False, 'hipster': False, 'romanti...",False,False,False,NaN,"{'dessert': False, 'latenight': False, 'lunch'..."
5,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"American (Traditional), Restaurants, Diners, B...",...,'none',False,'casual',True,"{'touristy': None, 'hipster': None, 'romantic'...",True,True,True,NaN,"{'dessert': False, 'latenight': False, 'lunch'..."
6,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"Food, Delis, Italian, Bakeries, Restaurants",...,u'none',True,u'casual',True,"{'romantic': False, 'intimate': False, 'touris...",True,True,True,u'average',"{'dessert': False, 'latenight': False, 'lunch'..."
7,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"Sushi Bars, Restaurants, Japanese",...,'full_bar',False,'casual',True,"{'touristy': False, 'hipster': False, 'romanti...",True,True,True,u'average',"{'dessert': True, 'latenight': None, 'lunch': ..."
8,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"Korean, Restaurants",...,u'none',True,u'casual',True,"{'touristy': False, 'hipster': False, 'romanti...",True,True,True,u'quiet',"{'dessert': False, 'latenight': False, 'lunch'..."
9,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",...,u'beer_and_wine',True,u'casual',True,"{'romantic': False, 'intimate': False, 'classy...",True,True,True,u'average',NaN


In [427]:
# Clean the 'RestaurantsPriceRange2' column
business_df['RestaurantsPriceRange2'] = pd.to_numeric(business_df['RestaurantsPriceRange2'], errors='coerce')
business_df['RestaurantsPriceRange2'] = business_df['RestaurantsPriceRange2'].fillna(business_df['RestaurantsPriceRange2'].mode()[0])

# Drop BusinessParking and GoodForMeal columns
business_df = business_df.drop(columns=['BusinessParking', 'GoodForMeal', 'Ambience'])
business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,categories,...,RestaurantsTakeOut,WiFi,Alcohol,Caters,RestaurantsAttire,RestaurantsReservations,GoodForKids,RestaurantsGoodForGroups,HasTV,NoiseLevel
0,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",...,True,u'free',u'none',True,NaN,False,False,False,False,NaN
1,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",...,True,u'no',u'none',True,u'casual',True,True,True,True,NaN
2,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"Pubs, Restaurants, Italian, Bars, American (Tr...",...,True,u'free',u'full_bar',True,u'casual',True,True,True,True,u'average'
3,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",...,True,u'no',u'none',True,'casual',True,True,True,True,NaN
4,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"Vietnamese, Food, Restaurants, Food Trucks",...,False,NaN,'none',False,NaN,True,False,False,False,NaN
5,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"American (Traditional), Restaurants, Diners, B...",...,True,u'no','none',False,'casual',True,True,True,True,NaN
6,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"Food, Delis, Italian, Bakeries, Restaurants",...,True,u'no',u'none',True,u'casual',True,True,True,True,u'average'
7,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"Sushi Bars, Restaurants, Japanese",...,True,'free','full_bar',False,'casual',True,True,True,True,u'average'
8,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"Korean, Restaurants",...,True,u'no',u'none',True,u'casual',True,True,True,True,u'quiet'
9,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",...,True,u'free',u'beer_and_wine',True,u'casual',True,True,True,True,u'average'


### Dealing with Missing Values from Remaining Features
There are still some missing values in the remaining features. We will need to fill in the missing values for the remaining features.

In [ ]:
# Check percentage of missing values
missing_percentage = business_df.isnull().mean()

# Print the column names with missing values percentage greater than 1%
missing_columns = missing_percentage[missing_percentage >= 0.01].index
missing_percentage = business_df[missing_columns].isnull().mean()

print(f"Features with 1% or more for missing data:\n{missing_percentage}")

Features with 1% or more for missing data:
WiFi                 0.270128
Alcohol              0.226991
RestaurantsAttire    0.246094
NoiseLevel           0.326276
dtype: float64


Since these features are categorical, we will need to fill in the missing values with the mode of the column. We will also need to convert the data types of some of the columns to make them easier to work with.

In [ ]:
columns_to_fill = ['WiFi', 'Alcohol', 'RestaurantsAttire', 'NoiseLevel']

# Fill missing values with the mode of the column
for col in columns_to_fill:
    business_df[col] = business_df[col].fillna(business_df[col].mode()[0])
    
# Check the first few rows of the DataFrame after filling missing values
business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,categories,...,RestaurantsTakeOut,WiFi,Alcohol,Caters,RestaurantsAttire,RestaurantsReservations,GoodForKids,RestaurantsGoodForGroups,HasTV,NoiseLevel
0,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",...,True,u'free',u'none',True,u'casual',False,False,False,False,u'average'
1,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",...,True,u'no',u'none',True,u'casual',True,True,True,True,u'average'
2,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,"Pubs, Restaurants, Italian, Bars, American (Tr...",...,True,u'free',u'full_bar',True,u'casual',True,True,True,True,u'average'
3,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",...,True,u'no',u'none',True,'casual',True,True,True,True,u'average'
4,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"Vietnamese, Food, Restaurants, Food Trucks",...,False,u'free','none',False,u'casual',True,False,False,False,u'average'
5,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"American (Traditional), Restaurants, Diners, B...",...,True,u'no','none',False,'casual',True,True,True,True,u'average'
6,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,"Food, Delis, Italian, Bakeries, Restaurants",...,True,u'no',u'none',True,u'casual',True,True,True,True,u'average'
7,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,"Sushi Bars, Restaurants, Japanese",...,True,'free','full_bar',False,'casual',True,True,True,True,u'average'
8,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,"Korean, Restaurants",...,True,u'no',u'none',True,u'casual',True,True,True,True,u'quiet'
9,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,"Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",...,True,u'free',u'beer_and_wine',True,u'casual',True,True,True,True,u'average'


### Extracting Features from the Categories Column
We will need to extract the features from the categories column. We will need to convert the categories into a one-hot encoded format so that we can use them in our analysis. We will also need to drop the original categories column.

In [439]:
business_df['categories_list'] = business_df['categories'].fillna('').apply(lambda x: [i.strip() for i in x.split(',')])

# Drop the original categories column
business_df = business_df.drop(columns=['categories'])
business_df.head(10)

,name,address,city,state,postal_code,latitude,longitude,stars,review_count,weekday_hours,...,WiFi,Alcohol,Caters,RestaurantsAttire,RestaurantsReservations,GoodForKids,RestaurantsGoodForGroups,HasTV,NoiseLevel,categories_list
0,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,40.0,...,u'free',u'none',True,u'casual',False,False,False,False,u'average',"[Restaurants, Food, Bubble Tea, Coffee & Tea, ..."
1,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,38.0,...,u'no',u'none',True,u'casual',True,True,True,True,u'average',"[Burgers, Fast Food, Sandwiches, Food, Ice Cre..."
2,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,43.0,...,u'free',u'full_bar',True,u'casual',True,True,True,True,u'average',"[Pubs, Restaurants, Italian, Bars, American (T..."
3,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,39.0,...,u'no',u'none',True,'casual',True,True,True,True,u'average',"[Ice Cream & Frozen Yogurt, Fast Food, Burgers..."
4,Vietnamese Food Truck,NaN,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,45.0,...,u'free','none',False,u'casual',True,False,False,False,u'average',"[Vietnamese, Food, Restaurants, Food Trucks]"
5,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,40.0,...,u'no','none',False,'casual',True,True,True,True,u'average',"[American (Traditional), Restaurants, Diners, ..."
6,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,45.0,...,u'no',u'none',True,u'casual',True,True,True,True,u'average',"[Food, Delis, Italian, Bakeries, Restaurants]"
7,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,40.0,...,'free','full_bar',False,'casual',True,True,True,True,u'average',"[Sushi Bars, Restaurants, Japanese]"
8,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,55.0,...,u'no',u'none',True,u'casual',True,True,True,True,u'quiet',"[Korean, Restaurants]"
9,Roast Coffeehouse and Wine Bar,10359 104 Street NW,Edmonton,AB,T5J 1B9,53.546045,-113.499169,4.0,40,40.0,...,u'free',u'beer_and_wine',True,u'casual',True,True,True,True,u'average',"[Coffee & Tea, Food, Cafes, Bars, Wine Bars, R..."
